# EyeWiki RAG Chatbot (Gemini / Colab)

日本語で眼科医師ロールの教育用チャットボットを作成します。EyeWiki をスクレイピングして RAG の知識ベースにし、Gemini API で回答します。

In [ ]:
!pip -q install google-generativeai beautifulsoup4 requests sentence-transformers faiss-cpu

In [ ]:
import os
import time
import re
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = input('Enter GOOGLE_API_KEY: ').strip()

genai.configure(api_key=GOOGLE_API_KEY)
MODEL_NAME = 'gemini-1.5-flash'


## 1. EyeWiki をスクレイピングしてドキュメントを作成

In [ ]:
BASE_URL = 'https://eyewiki.org/'
START_URL = 'https://eyewiki.org/Main_Page'
MAX_PAGES = 40  # 必要に応じて増減
REQUEST_DELAY = 1.0

def is_valid_article(href: str) -> bool:
    if not href:
        return False
    if not href.startswith('/'):  # relative links only
        return False
    if any(prefix in href for prefix in [
        '/Special:', '/Help:', '/File:', '/Category:', '/Talk:', '/User:', '/Template:'
    ]):
        return False
    return True

def extract_text(html: str) -> str:
    soup = BeautifulSoup(html, 'html.parser')
    content = soup.find('div', {'id': 'mw-content-text'})
    if not content:
        return ''
    for tag in content.find_all(['script', 'style', 'table']):
        tag.decompose()
    text = content.get_text(separator='
')
    text = re.sub(r'
{2,}', '
', text)
    return text.strip()

def collect_article_links(start_url: str, limit: int) -> list[str]:
    response = requests.get(start_url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    links = []
    for a in soup.select('div#mw-content-text a[href]'):
        href = a.get('href', '')
        if is_valid_article(href):
            url = urljoin(BASE_URL, href)
            links.append(url)
        if len(links) >= limit:
            break
    return links

def scrape_articles(urls: list[str]) -> list[dict]:
    docs = []
    for url in urls:
        time.sleep(REQUEST_DELAY)
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            text = extract_text(response.text)
            if text:
                docs.append({'url': url, 'text': text})
        except requests.RequestException as exc:
            print(f'Failed to fetch {url}: {exc}')
    return docs

article_urls = collect_article_links(START_URL, MAX_PAGES)
documents = scrape_articles(article_urls)
print(f'Scraped {len(documents)} documents')


## 2. 埋め込みと RAG インデックスを作成

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def chunk_text(text: str, chunk_size: int = 800, overlap: int = 100) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        start = end - overlap
        if start < 0:
            start = 0
    return chunks

chunk_texts = []
chunk_meta = []
for doc in documents:
    for chunk in chunk_text(doc['text']):
        chunk_texts.append(chunk)
        chunk_meta.append({'url': doc['url']})

embeddings = embedder.encode(chunk_texts, normalize_embeddings=True)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.array(embeddings, dtype='float32'))
print(f'Indexed {len(chunk_texts)} chunks')


## 3. RAG チャット関数

In [ ]:
def retrieve_context(query: str, top_k: int = 4) -> tuple[str, list[str]]:
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(q_emb, dtype='float32'), top_k)
    contexts = []
    urls = []
    for idx in indices[0]:
        contexts.append(chunk_texts[idx])
        urls.append(chunk_meta[idx]['url'])
    return '\n\n'.join(contexts), list(dict.fromkeys(urls))

def ophthalmology_chat(question: str) -> str:
    context, urls = retrieve_context(question)
    prompt = (
        'あなたは眼科医師として医学生向けに説明する専門家です。\n'
        '以下の参考情報を根拠に、日本語で簡潔かつ正確に回答してください。\n'
        '必要なら箇条書きを使い、最後に参考URLを列挙してください。\n\n'
        f'参考情報:\n{context}\n\n'
        f'質問: {question}\n'
    )
    model = genai.GenerativeModel(MODEL_NAME)
    response = model.generate_content(prompt)
    answer = response.text
    sources = '\n'.join(f'- {url}' for url in urls)
    return f"{answer}\n\n参考URL:\n{sources}"

ophthalmology_chat('緑内障の簡単な概要を教えてください')


## 4. 対話ループ (必要な場合)

In [ ]:
def chat_loop():
    print('終了するには空入力')
    while True:
        question = input('質問: ').strip()
        if not question:
            break
        print(ophthalmology_chat(question))

# chat_loop()
